# 0 · Setup: get ready for the course

**AI-Augmented Productivity for Finance · IESE MiF**

Welcome. This notebook verifies your machine is ready. Work top to bottom: read each text cell, then run each code cell with **Shift+Enter**. If VS Code asks you to *select a kernel*, pick your **aifinance** environment (or the `.venv` you created in `setup/SETUP.md`).

> **How this course works.** Each session is one notebook (`01` to `05`), and **everything happens inside the notebooks**: teaching, demonstrations, and labs. First part of each = the ideas, with runnable demonstrations. Second part = **the lab**: between `### START CODE HERE ###` and `### END CODE HERE ###` markers you'll find code with **gaps**: every `None` and every `[BRACKETED BLANK]` is yours to fill (the Python structure is given; the *finance thinking* is the exercise). Then run the ✅ *self-check* cell below each exercise. When it prints "All checks passed", move on. Stuck? Ask **Claude Code** (the ✱ panel in VS Code): using it well *is* the course.

## The two credentials you need: both from Day 1

| Credential | Powers | Where from | Cost |
|---|---|---|---|
| **Claude Pro plan** | Claude Code: your copilot in every lab | claude.ai subscription (from the first pre-course) | ~$20/one month. The FREE account does **not** run Claude Code |
| **Anthropic API key** (an API, application programming interface, is the channel your own code calls a service through) | The notebook cells where *your own code* calls Claude: **used from Session 1 onward** | console.anthropic.com → $5 credit → API Keys | you'll use €1–2 total |

Full instructions: `setup/claude-code-setup.md`. Your API key goes into the `.env` file (`cp .env.example .env`, then edit). **Never** commit or paste the key anywhere.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - see notebooks/00-setup.ipynb'}")

## Environment check

The cell below runs the course's full environment checker. Everything required should show `[ OK ]`.

> **Golden notebook rule:** a cell's output shows the LAST time it ran, not the current truth, and the kernel snapshots your environment when it starts. If you install packages after starting: **Restart → Run All**. The same applies if the course files change under you (a `git pull`): the first cell of every notebook now reloads the course toolkit for you, but a **Restart** is always the reliable cure for anything unexplained. Whenever a notebook disagrees with your terminal, restart the kernel first and re-run before believing either.

> **Reading long answers.** Model output is shown with `llm.show(...)`, which renders it as formatted, wrapped markdown so you always see the whole answer. Never judge an answer you can only see the first 150 characters of.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "setup" / "check_setup.py")],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

### Any `[FAIL] package: ...` lines above? Run the rescue cell below.

With several Python environments on one laptop (bootcamp leftovers, venvs,
conda), packages easily land in a different environment than the kernel
running this notebook. The cell below is mismatch-proof: it installs the
course requirements into **exactly the kernel you selected**: then
**Restart → Run All** and the checks go green.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(ROOT / "requirements.txt")], capture_output=True, text=True)
print(r.stderr[-300:] if r.returncode else
      "Installed into THIS kernel ✅  Now: Restart → Run All.")

## Can you reach SEC EDGAR — the U.S. Securities and Exchange Commission's public filing database?

The course runs on free, public SEC data. This fetches Apple's official company ID from the SEC: your first real financial-infrastructure call.

In [ ]:
from toolkit import edgar

print("Apple's SEC CIK number:", edgar.cik_for("AAPL"))
print("Latest Apple 10-K:", edgar.recent_filings("AAPL", forms=["10-K"], limit=1)[0]["filed"])

## Can your code talk to Claude?

This must work before Session 1: the very first lab calls Claude from these cells.

In [ ]:
if HAS_KEY:
    from toolkit import llm
    reply = llm.ask("Reply with exactly: OK", max_tokens=200)
    print("Claude says:", reply.strip())
    print("Model:", llm.default_model())
    print("✅ your API key works" if "OK" in reply.upper()
          else "⚠️ unexpected reply above; the key works but check the message")
else:
    # Self-diagnosis: find out WHY the key was not picked up.
    print("❌ No API key found. Diagnosis:\n")
    env = ROOT / ".env"
    print(f"1. This notebook looked for: {env}")
    if not env.exists():
        print("   -> that file does NOT exist. Two likely reasons:")
        stray = sorted(str(q) for q in ROOT.glob(".env*") if q.name != ".env")
        others = sorted(str(q) for q in ROOT.parent.glob("*/.env")) if ROOT.parent != ROOT else []
        if any(s.endswith(".env.txt") for s in stray):
            print("      (a) Windows added .txt: rename '.env.txt' to '.env'")
        print("      (b) you are not in the course folder, or .env sits elsewhere.")
        print(f"      Files here starting with .env: {stray or 'none'}")
        if others:
            print(f"      A .env exists in a NEIGHBOURING folder: {others}")
            print("      -> open the folder that contains requirements.txt (File > Open Folder)")
    else:
        raw = env.read_text()
        line = next((l for l in raw.splitlines()
                     if l.strip().startswith("ANTHROPIC_API_KEY")), None)
        print("   -> the file exists.")
        if line is None:
            print("2. It has no ANTHROPIC_API_KEY line at all. Add: ANTHROPIC_API_KEY=sk-ant-...")
        else:
            value = line.split("=", 1)[1].strip() if "=" in line else ""
            if not value:
                print("2. The ANTHROPIC_API_KEY line is EMPTY. Paste your key after the '=' (no spaces, no quotes).")
            elif value.startswith(("'", '"')):
                print("2. Your key is wrapped in quotes. Remove them: ANTHROPIC_API_KEY=sk-ant-...")
            else:
                print(f"2. A key IS present in the file ({len(value)} characters).")
                shell_var = os.environ.get("ANTHROPIC_API_KEY", None)
                if shell_var is not None:
                    print("   -> BUT this session already has an ANTHROPIC_API_KEY variable set")
                    print(f"      to something unusable ({len(shell_var)} characters), and a .env")
                    print("      file never overrides a variable that already exists.")
                    print("   FIX: close every VS Code terminal, then Restart the kernel and Run All.")
                else:
                    print("   -> So this kernel started BEFORE you saved the file.")
                    print("   FIX: press Restart above, then Run All. That is all it needs.")
                print()
                print("   (The VS Code popup about \"terminal environment injection\" is NOT the")
                print("    cause - this notebook reads .env in code and ignores that setting.)")
    print("\n3. Whatever the cause: after fixing, always Restart the kernel and Run All -")
    print("   a running kernel never notices a file you edited afterwards.")


## And the ✱ Claude Code panel?

Two manual checks (30 seconds):

1. Click the **✱ Claude icon** (editor toolbar top-right, or "✱ Claude Code" in the status bar). Type `/status`: it should show your **Pro** plan. ⚠️ If your chat shows *GPT/Gemini* models or a *credits* counter, you are in a **different extension**, such as Copilot. Locate the ✱ panel instead.
2. Ask it, with nothing selected: *"What is the one rule that always applies in this repo?"* It should answer from this repo's `CLAUDE.md`: **never present a number you haven't verified against the source data.**

That's the course motto. You're ready: open `01-prompting.ipynb`.